# Smart Factory FAQ - Bilingual RAG Chatbot
### Arabic + English | Cohere Embeddings + FAISS + LangChain
---
يدعم الاسئله بالعربي والانجليزي والاجابات بالاتنين.


In [38]:
# !pip install cohere langchain langchain-cohere langchain-community faiss-cpu
import os, json, warnings
import cohere
from langchain_core.documents  import Document
from langchain_core.prompts    import PromptTemplate
from langchain_community.vectorstores import FAISS
from langchain_core.embeddings import Embeddings
from langchain_cohere          import ChatCohere
from langchain_core.messages   import HumanMessage, AIMessage
from collections import Counter
import os
from dotenv import load_dotenv
warnings.filterwarnings('ignore')
print('Libraries loaded')

Libraries loaded


In [39]:
COHERE_API_KEY = 'YOUR_API_KEY_HERE'   # <- put your Cohere API key here
os.environ['COHERE_API_KEY'] = COHERE_API_KEY
print('API Key set')

API Key set


In [40]:
class CohereEmbeddingsFixed(Embeddings):
    """
    Multilingual embeddings - supports Arabic and English with the same model.
    Model: embed-multilingual-v3.0
    """
    def __init__(self, api_key, model='embed-multilingual-v3.0'):
        self.model  = model
        self.client = cohere.Client(api_key)
        self.client.embed(texts=['test'], model=self.model, input_type='search_document')
        print('Cohere multilingual embeddings ready')

    def embed_documents(self, texts):
        if not texts:
            return []
        res = self.client.embed(texts=texts, model=self.model, input_type='search_document')
        return [list(v) for v in res.embeddings]

    def embed_query(self, text):
        res = self.client.embed(texts=[text], model=self.model, input_type='search_query')
        return list(res.embeddings[0])

print('CohereEmbeddingsFixed defined')

CohereEmbeddingsFixed defined


In [41]:
file_path = 'smart_factory_faq_v3.json'  # <- update path if needed

with open(file_path, 'r', encoding='utf-8') as f:
    faq = json.load(f)

print(f'Loaded {len(faq)} FAQ entries')

# Language distribution
lang_counts = Counter(item['language'] for item in faq)
print('Language distribution:', dict(lang_counts))

# Show one sample per language
for lang in ['en', 'ar', 'bilingual']:
    sample = next(i for i in faq if i['language'] == lang)
    print(f"\n[{lang.upper()}] Q: {sample['question'][:80]}")
    print(f"       A: {sample['answer'][:80]}...")

Loaded 64 FAQ entries
Language distribution: {'en': 32, 'ar': 16, 'bilingual': 16}

[EN] Q: Why is the machine showing high voltage readings?
       A: 1. Check volt_lag1 and volt_lag3 for sudden spike pattern
2. Review volt_diff — ...

[AR] Q: ما سبب ارتفاع قراءات الجهد الكهربائي في الماكينة؟
       A: 1. افحص volt_lag1 و volt_lag3 للكشف عن نمط الارتفاع المفاجئ
2. راجع volt_diff — ...

[BILINGUAL] Q: Why is the machine showing high voltage readings?
ما سبب ارتفاع قراءات الجهد الك
       A: 1. Check volt_lag1 and volt_lag3 for sudden spike pattern
2. Review volt_diff — ...


In [42]:
docs = []

for item in faq:
    content = f"Question: {item['question']}\n\nAnswer: {item['answer']}"
    docs.append(Document(
        page_content=content,
        metadata={
            'answer':        item['answer'],
            'fault':         item['fault'],
            'sensor_column': item['sensor_column'],
            'severity':      item['severity'],
            'language':      item['language'],
            'question_type': item['question_type'],
        }
    ))

print(f'{len(docs)} documents prepared')

64 documents prepared


In [43]:
print('Building FAISS vector store ...')
load_dotenv(override=True)  # Load environment variables from .env file

COHERE_API_KEY = os.getenv("COHERE_API_KEY")

if not COHERE_API_KEY:
    raise ValueError("COHERE_API_KEY environment variable is not set")

embeddings  = CohereEmbeddingsFixed(api_key=COHERE_API_KEY)
vectorstore = FAISS.from_documents(docs, embeddings)
print(f'FAISS ready - {len(docs)} vectors indexed')

Building FAISS vector store ...
Cohere multilingual embeddings ready
FAISS ready - 64 vectors indexed


In [44]:
retriever = vectorstore.as_retriever(
    search_type='mmr',
    search_kwargs={
        'k':       8,    # number of final results
        'fetch_k': 30,   # candidates before MMR filtering
    }
)
print('Retriever ready (MMR, k=8)')

Retriever ready (MMR, k=8)


In [45]:
llm = ChatCohere(
    model='command-r-plus-08-2024',
    temperature=0,
    cohere_api_key=COHERE_API_KEY,
)
print('LLM ready')

LLM ready


In [46]:
PROMPT_TEMPLATE = (
    'You are an intelligent maintenance assistant for a Smart Factory Monitoring Platform.\n'
    'You are a bilingual assistant - you support both Arabic and English.\n'
    '\n'
    'CRITICAL RULES:\n'
    '1. Use ONLY the provided CONTEXT. Do not use external knowledge.\n'
    '2. If the answer is not in the context, respond ONLY with:\n'
    "   'No relevant data found. | لا توجد بيانات ذات صلة.'\n"
    '3. Never hallucinate sensor values or thresholds.\n'
    '4. Detect the question language and respond in the SAME language:\n'
    '   - English question -> steps in English only.\n'
    '   - Arabic question  -> steps in Arabic only.\n'
    '   - Mixed question   -> steps in English first, then Arabic.\n'
    '5. ALWAYS write exactly 5 steps. No more, no less.\n'
    '6. Labels (Fault, Sensor, Severity, Steps) ALWAYS stay in English. Do NOT translate them.\n'
    '\n'
    'CONTEXT:\n'
    '{context}\n'
    '\n'
    'CONVERSATION HISTORY:\n'
    '{history}\n'
    '\n'
    'QUESTION:\n'
    '{question}\n'
    '\n'
    'RESPONSE FORMAT (STRICT - DO NOT CHANGE THE LABELS):\n'
    'Fault:    <fault name>\n'
    'Sensor:   <sensor column>\n'
    'Severity: <Critical | High | Medium>\n'
    'Steps:\n'
    '1. <step>\n'
    '2. <step>\n'
    '3. <step>\n'
    '4. <step>\n'
    '5. <step>\n'
)

prompt = PromptTemplate(
    input_variables=['context', 'history', 'question'],
    template=PROMPT_TEMPLATE,
)
print('Bilingual prompt ready')

Bilingual prompt ready


In [47]:
def build_context(retrieved_docs):
    """
    Builds context string from retriever results.
    Includes metadata: fault, sensor, severity, language.
    """
    parts = []
    for doc in retrieved_docs:
        m = doc.metadata
        header = (
            f"[Fault: {m.get('fault','-')} | "
            f"Sensor: {m.get('sensor_column','-')} | "
            f"Severity: {m.get('severity','-')} | "
            f"Lang: {m.get('language','-')}]"
        )
        parts.append(f"{header}\n{doc.page_content}")
    return '\n\n'.join(parts)

print('build_context() ready')

build_context() ready


In [48]:
from datetime import datetime

history      = []
question_num = 0

print('=' * 57)
print('  Smart Factory Maintenance Chatbot')
print('  Supports Arabic and English questions')
print('  يدعم الاسئله بالعربي والانجليزي')
print("  Type 'exit' to quit | اكتب 'exit' للخروج")
print('=' * 57)

while True:
    user_input = input('\nYou | انت: ').strip()

    if user_input.lower() in ['exit', 'quit', 'خروج']:
        print('Goodbye! | مع السلامة!')
        break
    if not user_input:
        continue

    question_num += 1
    timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

    # 1. Retrieve relevant FAQ docs
    retrieved = retriever.invoke(user_input)

    # 2. Build context with metadata
    context = build_context(retrieved)

    # 3. Last 6 messages (3 turns) as history string
    history_str = '\n'.join([
        f"{'User' if isinstance(m, HumanMessage) else 'Bot'}: {m.content}"
        for m in history[-6:]
    ]) or 'No previous conversation.'

    # 4. Format and call LLM
    final_prompt = prompt.format(
        context  = context,
        history  = history_str,
        question = user_input,
    )
    response = llm.invoke(final_prompt)
    answer   = response.content

    # 5. Print formatted output
    top_fault = retrieved[0].metadata.get('fault', '-')
    print(f'\n[Q{question_num}] [{timestamp}]')
    print(f'You:  {user_input}')
    print(f'Top fault matched: {top_fault}')
    print(f'\nBot:\n{answer}')
    print('-' * 57)

    # 6. Save to history
    history.append(HumanMessage(content=user_input))
    history.append(AIMessage(content=answer))

  Smart Factory Maintenance Chatbot
  Supports Arabic and English questions
  يدعم الاسئله بالعربي والانجليزي
  Type 'exit' to quit | اكتب 'exit' للخروج
Goodbye! | مع السلامة!
